# Post-training attention and probe visualizations

Training logs numerical attention activations and probe scalars at full cadence. This notebook downloads exact W&B history steps and recreates the former images locally. Downloaded tables are cached under `notebooks/.cache/`; optional exports go to the gitignored `notebooks/figures/` directory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from notebooks.utils import (
    DEFAULT_ENTITY, DEFAULT_PROJECT, fetch_attention_tables, fetch_run_config,
    fetch_step_metrics, probe_metric_keys, value_alignment_from_metrics,
    value_alignment_keys,
)
from src.analysis.training_visualizations import (
    attention_table_to_array, plot_attention_alignment, plot_attention_heatmaps,
    plot_probe_heatmap, plot_value_alignment, probe_layout_from_config,
    probe_matrix_from_metrics, save_figures,
)

## Select a run and exact logged steps

`weights` in the W&B table key means attention activations, not learned parameter weights. Validation is the default because attention dropout is disabled there.

In [ ]:
ENTITY = DEFAULT_ENTITY
PROJECT = DEFAULT_PROJECT
RUN_ID = "replace-me"
LAYER = "L1"
SPLIT = "val"
ATTENTION_STEPS = [0, 25, 100, 500, 1000, 1975]
PROBE_STEPS = [0, 100, 500, 1000]
SAVE_FIGURES = False
OUTPUT_DIR = Path("notebooks/figures") / RUN_ID

config = fetch_run_config(RUN_ID, entity=ENTITY, project=PROJECT)
teacher_cfg = config["teacher"]
student_cfg = config["student"]
config["misc"]["wandb"].get("tags", [])

## Attention heatmaps and span alignment

In [ ]:
tables = fetch_attention_tables(
    RUN_ID, ATTENTION_STEPS, layer=LAYER, split=SPLIT,
    entity=ENTITY, project=PROJECT,
)
span_lengths = teacher_cfg.get("span_lengths")
stride = teacher_cfg.get("stride")
context_length = None
if span_lengths:
    context_length = ((len(span_lengths) - 1) * stride + span_lengths[-1]
                      if stride is not None else sum(span_lengths))

for step, table in tables.items():
    attention = attention_table_to_array(table)
    figures = plot_attention_heatmaps(attention, step=step, split=SPLIT)
    if context_length is not None:
        figures.update(plot_attention_alignment(
            attention, list(span_lengths), context_length, step, SPLIT, stride
        ))
    if SAVE_FIGURES:
        save_figures(figures, OUTPUT_DIR / "attention", f"{LAYER}_{SPLIT}_step{step}")
    for name, figure in figures.items():
        print(f"step {step}: {name}")
        display(figure)
        plt.close(figure)

## Value-projection alignment

This section applies only to runs with `attention_disentanglement=true`. The heatmaps are reconstructed from numerical norm and cosine histories.

In [ ]:
if student_cfg.get("attention_disentanglement", False) and "span_lengths" in teacher_cfg:
    num_heads = int(student_cfg["num_heads"])
    num_teacher = int(teacher_cfg.get("window", len(teacher_cfg["span_lengths"])))
    norm_keys, cosine_keys = value_alignment_keys(num_heads, num_teacher, LAYER, SPLIT)
    rows = fetch_step_metrics(
        RUN_ID, ATTENTION_STEPS, norm_keys + cosine_keys,
        entity=ENTITY, project=PROJECT,
    )
    for step, row in rows.items():
        norms, cosine = value_alignment_from_metrics(
            row, num_heads, num_teacher, LAYER, SPLIT
        )
        figures = plot_value_alignment(norms, cosine, step, SPLIT)
        if SAVE_FIGURES:
            save_figures(figures, OUTPUT_DIR / "value", f"{LAYER}_{SPLIT}_step{step}")
        for name, figure in figures.items():
            print(f"step {step}: {name}")
            display(figure)
            plt.close(figure)
else:
    print("No logged per-head value alignment is available for this run.")

## Probe summary heatmaps

Accuracy and excess-NLL layer×slot grids are reconstructed from existing scalar histories. No probe image media is required.

In [ ]:
probe_mode = config.get("misc", {}).get("probe", {}).get("mode", "off")
if probe_mode != "off":
    layout = probe_layout_from_config(config)
    requests = [
        (level, offset, metric)
        for level in range(len(layout.slots_per_level))
        for offset in layout.offsets
        for metric in ("acc", "excess_nll")
    ]
    keys = [
        key
        for level, offset, metric in requests
        for key in probe_metric_keys(
            level, offset, metric, layout.num_layers,
            layout.slots_per_level[level], SPLIT,
        )
    ]
    rows = fetch_step_metrics(
        RUN_ID, PROBE_STEPS, keys, entity=ENTITY, project=PROJECT
    )
    for step, row in rows.items():
        for level, offset, metric in requests:
            matrix = probe_matrix_from_metrics(
                row, level, offset, metric, SPLIT, layout.num_layers,
                layout.slots_per_level[level],
            )
            if np.isnan(matrix).all():
                continue
            figure = plot_probe_heatmap(matrix, level, offset, metric, step, SPLIT)
            name = f"level{level}_k{offset:+d}_{metric}"
            if SAVE_FIGURES:
                save_figures({name: figure}, OUTPUT_DIR / "probe", f"{SPLIT}_step{step}")
            print(f"step {step}: {name}")
            display(figure)
            plt.close(figure)
else:
    print("Probe mode is off for this run.")